In [ ]:
import pandas as pd
sitefinity = pd.read_csv('Sitefinity.csv')
reach = pd.read_csv('reach.csv')

reachList = [i.lower() for i in reach["From REACH"]]
sitefinityList = [i.lower() for i in sitefinity["Sitefinity"]]

sitefinityNotInREACH = []
reachNotInSitefinity = []

for i in sitefinityList:
    if i not in reachList:
        sitefinityNotInREACH.append(i)

for i in reachList:
    if i not in sitefinityList:
        reachNotInSitefinity.append(i)

df = pd.DataFrame(reachNotInSitefinity, columns=["title"])
df.to_csv("REACHH.csv", index=False)

In [ ]:
# Generate collection - link varient
month = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
year = ["2020", "2021", "2022", "2023", "2024"]

for i in year:
    for x in month:
        schema = {
            "version": "0.1.0",
            "layout": "link",
            "page": {
              "title": f"Better Cities {x[0].upper()}{x[1:]} {i}",
              "ref": f"/research-publications/publications-library/{i}-{x}",
              "category": "Better Cities",
              "date": f"1 {x[0].upper()}{x[1:]} {i}"
            },
            "content": []
        }
    
        # Create the directory if it doesn't exist
        os.makedirs("docs", exist_ok=True)
        
        # Create the file path using os.path.join
        file_path = os.path.join("docs", f"{i}-{x}.json")
        
        # Add index to name to prevent duplication
        if os.path.exists(file_path):
            file_path = os.path.join("docs", f"{i}-{x}.json")
        
        # Create json file
        with open(f"{file_path}", 'w+', encoding='utf-8') as f:
            json.dump(schema, f, ensure_ascii=False, indent=2)

In [ ]:
for i in range(0, len(data)):
    page = requests.get(data[i]["url"])
    soup = BeautifulSoup(page.content, "html.parser")
    
    content = soup.find("div", class_="sf_2cols_1in_75")
    content1 = content.find_all("div", class_="row")
    data[i]["html"] = content1

df = pd.DataFrame.from_dict(data, orient='index')
df.to_csv("better-cities.csv", index=False)

In [9]:
import requests
from requests.auth import HTTPBasicAuth
from urllib.parse import urljoin, urlparse
from dotenv import load_dotenv
import os
import json
from bs4 import BeautifulSoup

########################################
#             Side Quest #4            #
########################################

load_dotenv()
password = os.getenv("mainsite")
domain = "https://staging.d9vo48leqc1gd.amplifyapp.com"
url = domain + "/sitemap.json"

page = requests.get(url, auth=HTTPBasicAuth('user', password))
pageJson = json.loads(page.text)

path = []
counter = 0
for index, value in enumerate(pageJson['children']):
    try:
        if 'ref' in pageJson['children'][index]:
            path.append(pageJson['children'][index]['ref'])
        else:
            path.append(pageJson['children'][index]['permalink'])
        if 'children' in pageJson['children'][index]:
            for i in range(0, len(pageJson['children'][index]['children'])):
                if 'ref' in pageJson['children'][index]['children'][i]:
                    path.append(pageJson['children'][index]['children'][i]['ref'])
                else:
                    path.append(pageJson['children'][index]['children'][i]['permalink'])
                if 'children' in pageJson['children'][index]['children'][i]:
                    for c in range(0, len(pageJson['children'][index]['children'][i]['children'])):
                        if 'ref' in pageJson['children'][index]['children'][i]['children'][c]:
                            path.append(pageJson['children'][index]['children'][i]['children'][c]['ref'])
                        else:
                            path.append(pageJson['children'][index]['children'][i]['children'][c]['permalink'])
                        if 'children' in pageJson['children'][index]['children'][i]['children'][c]:
                            for y in range(0, len(pageJson['children'][index]['children'][i]['children'][c]['children'])):
                                if 'ref' in pageJson['children'][index]['children'][i]['children'][c]['children'][y]:
                                    path.append(pageJson['children'][index]['children'][i]['children'][c]['children'][y]['ref'])
                                else:
                                    path.append(pageJson['children'][index]['children'][i]['children'][c]['children'][y]['permalink'])
    except Exception as e:
        pass
print(len(path))

375


In [60]:
print(path)

['/advisory-and-partners', '/advisory-and-partners/advisory-services', '/advisory-and-partners/key-partners', '/capability-development', '/capability-development/international-programmes', '/capability-development/international-programmes/jspp21', '/capability-development/international-programmes/mfa-singapore-cooperation-programmes-(scp)---topical-programmes', '/capability-development/international-programmes/past-programmes', '/capability-development/international-programmes/sgunhilugp', '/capability-development/international-programmes/singapore-pacific-resilience-and-knowledge-sharing-(sparks)', '/capability-development/local-milestone-programmes', '/capability-development/local-milestone-programmes/city-excel', '/capability-development/local-milestone-programmes/edge', '/capability-development/local-milestone-programmes/lugp', '/contact-us', '/events', '/events/clc-lecture-series', '/events/lecture/5th-singapore-shanghai-dialogue-on-urban-governance', '/events/lecture/2040-seoul-p

In [62]:
from requests.exceptions import SSLError
from urllib3.exceptions import NewConnectionError

domain = "https://staging.d9vo48leqc1gd.amplifyapp.com"

collection = ["clc-lecture-series", "clc-webinar-series", "media-mentions"]
for i, v in enumerate(path):
    internalLinks = []

    if "https://" in v or "http://" in v:
        try:
            resp = requests.get(i, allow_redirects=True)
            if resp.status_code == 404:
                print("href:", v, "404!")
        except NewConnectionError:
            print("href:", v, "404!")
        except requests.exceptions.RequestException as e:
            print("href:", v, "404!")
    else:
        url = domain + v
        page = requests.head(url, auth=HTTPBasicAuth('user', password), allow_redirects=True)
        
        if page.url != domain + "/404.html":
            if "/files" in v or "/images" in v:
                continue
            elif "/files" not in v or "/images" not in v:
                if v.split("/")[-1] not in collection:
                    page = requests.get(url, auth=HTTPBasicAuth('user', password), allow_redirects=True)
                    soup = BeautifulSoup(page.content, "html.parser")
            
                    try:
                        content = soup.find("div", class_ = "col-span-12 flex flex-col gap-16 lg:col-span-9 lg:mr-24")
                        # print(v)
                        links = content.find_all("a")
                    except AttributeError:
                        try:
                            content = soup.find("div", class_ = "grid grid-cols-1 gap-10 md:gap-7 lg:gap-x-16 lg:gap-y-12")
                            # print(v)
                            links = content.find_all("a")
                        except AttributeError:
                            content = soup.find("div", class_ = "col-span-12 flex flex-col gap-16 max-w-[54rem]")
                            # print(v)
                            links = content.find_all("a")

                    for index, value in enumerate(links):
                        try:
                            if "https://" in value["href"] or "http://" in value["href"]:
                                try:
                                    resp = requests.get(value["href"], allow_redirects=True)
                                    if resp.status_code == 404:
                                        print("Link:", url, "href:", value["href"], "404!")
                                except NewConnectionError:
                                    print("Link:", url, "href:", value["href"], "404!")
                                except requests.exceptions.RequestException as e:
                                    print("Link:", url, "href:", value["href"], "404!")
                                    
                            if "undefined" in value["href"]:
                                print("Link:", domain+v, "href:", value["href"], "404!")
                                continue
                            if value["href"][0] != "#" and "mailto:" not in value["href"] and "tel:" not in value["href"]:
                                page = requests.head(domain+value["href"] if "https" not in value["href"] or "http" not in value["href"] else value["href"], auth=HTTPBasicAuth('user', password), allow_redirects=True)

                                if "PageNotFound" in page.url or page.url == domain + "/404.html" or "closepage" in page.url or page.url == domain + "/undefined":
                                    print("Link:", domain+v, "href:", value["href"], "Redirected:", page.url, "404!")
                                    
                                if page.url != domain + "/404.html":
                                    # print(domain+value["href"], "Not 404!")
                                    continue 
                            else:
                                continue
                        except SSLError:
                            print("Link:", domain+v, "href:", value["href"], page.url, "404!")
                        except requests.exceptions.Timeout:
                            print("Link:",domain+v, "href:", value["href"], page.url, "404!")
                        except Exception as e:
                            print(e)
        else:
            print(v, "404!")        

Link: https://staging.d9vo48leqc1gd.amplifyapp.com/advisory-and-partners/advisory-services href: https://www.mti.gov.sg/amaravati/EN/Pages/Home.aspx Redirected: https://www.mti.gov.sg/Cwp/PageNotFound.aspx 404!
Link: https://staging.d9vo48leqc1gd.amplifyapp.com/advisory-and-partners/key-partners href: https://www.sde.nus.edu.sg/ 404!
Link: https://staging.d9vo48leqc1gd.amplifyapp.com/advisory-and-partners/key-partners href: https://www.sde.nus.edu.sg/ https://global.si.re.kr/ 404!
Link: https://staging.d9vo48leqc1gd.amplifyapp.com/advisory-and-partners/key-partners href: undefined 404!
Link: https://staging.d9vo48leqc1gd.amplifyapp.com/events/lecture/beyond-administrative-delimitations href: undefined 404!
Link: https://staging.d9vo48leqc1gd.amplifyapp.com/events/lecture/bridging-the-public-private-divide-to-create-great-cities href: undefined 404!
Link: https://staging.d9vo48leqc1gd.amplifyapp.com/events/lecture/bringing-town-planning-to-the-future href: undefined 404!
Link: https://s

ConnectionError: HTTPSConnectionPool(host='staging.d9vo48leqc1gd.amplifyapp.com', port=443): Max retries exceeded with url: /who-we-are/annual-reports (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x16815e090>: Failed to resolve 'staging.d9vo48leqc1gd.amplifyapp.com' ([Errno 8] nodename nor servname provided, or not known)"))

In [ ]:
try:
    print(path[225])
    resp = requests.get(path[225], allow_redirects=True)
    if resp.status_code == 404:
        print(path[225], "404!")
except NewConnectionError:
    print(path[225], "404!")
except requests.exceptions.RequestException as e:
    print(path[225], "404!")

In [ ]:
print(path[286])